# Step 8 · RL (rules) and RLHF (reward model) with GRPO

The model writes several answers per question, a reward scores them, GRPO moves toward the better ones. Reward = rules → RL. Reward = a model trained on human preferences → RLHF.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
from lab import jobs
APPROVE_IF_NEEDED = True   # RL needs an approved reward version; approves the newest draft only if none is approved
approved = [r["id"] for r in registry.rewards() if r["approved"]]
if not approved and APPROVE_IF_NEEDED:
    registry.update("rewards", registry.rewards()[0]["id"], approved=1)
    approved = [registry.rewards()[0]["id"]]
PARENT = latest("sft") or BASE
cfg = dict(jobs.RL_PRESETS["Tiny (smoke test)"], reward_source="rules")
print("RL from", PARENT, "with", approved[0], cfg)

In [ ]:
rid = jobs.start("rl", PARENT, latest_ds("rl_train"), latest_ds("rl_val"), cfg, reward_id=approved[0], wait=True)
run = registry.run(rid); print(run["status"], "→", run["output_model_id"], run["error"] or "")
pd.DataFrame(jobs.read_jsonl(run["samples_path"]))[["step", "question", "answer", "total"]].tail(8)

### RLHF — same GRPO, reward from the reward model (train one in notebook 10 first)

In [ ]:
RM = latest("rm")
if RM:
    cfg = dict(jobs.RL_PRESETS["Tiny (smoke test)"], reward_source="rm", rm_id=RM)
    rid = jobs.start("rlhf", PARENT, latest_ds("rl_train"), latest_ds("rl_val"), cfg, wait=True)
    run = registry.run(rid); print(run["status"], "→", run["output_model_id"], run["error"] or "")
    display(pd.DataFrame(jobs.read_jsonl(run["samples_path"]))[["question", "answer", "total"]].tail(8))
else:
    print("No reward model yet — run notebooks 09 and 10, then come back.")